In [1]:
import os
import ray
import json
import pickle
from dynaconf import Dynaconf
from tqdm.notebook import tqdm
from utils import check_path, convert_np_arrays, flatten_dict, env_creator
from ray.tune.logger import JsonLogger
from algorithms_with_statistics.ddqn_pber import DDQNWithMPBERAndLogging
from replay_buffer.mpber import MultiAgentPrioritizedBlockReplayBuffer
from ray.tune.registry import register_env

pygame 2.5.2 (SDL 2.28.2, Python 3.9.16)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# Init Ray
ray.init(
    num_cpus=5, num_gpus=1,
    include_dashboard=False,
    _system_config={"maximum_gcs_destroyed_actor_cached_count": 20},
)

2024-04-06 14:52:12,290	INFO worker.py:1673 -- Started a local Ray instance.


Python version:,3.9.16
Ray version:,2.8.0


In [3]:
import mlflow
from mlflow.exceptions import MlflowException
from func_timeout import FunctionTimedOut
from botocore.exceptions import ConnectionClosedError

In [4]:
import datetime
# Config path
env_name = "Breakout"
run_name = int(datetime.datetime.now().timestamp())
log_path = "/home/seventheli/data/BER/experiments/logging/%s" % env_name
checkpoint_path = "/home/seventheli/data/BER/experiments/checkpoints/%s" % env_name
# Set mlflow
mlflow.set_tracking_uri("http://localhost:9999")
mlflow.set_experiment(experiment_name=env_name)
mlflow_client = mlflow.tracking.MlflowClient()

In [5]:
# Check path available
import shutil
check_path(log_path)
log_path = str(os.path.join(log_path, str(run_name)))
check_path(log_path)
if os.path.exists(log_path):
    shutil.rmtree(log_path)
check_path(log_path)
check_path(checkpoint_path)
checkpoint_path = os.path.join(checkpoint_path, str(run_name))
check_path(checkpoint_path)
if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
check_path(checkpoint_path)

In [6]:
# Set hyper parameters
setting = "./settings/ddqn_atari.yml"
setting = Dynaconf(envvar_prefix="DYNACONF", settings_files=setting)

hyper_parameters = setting.hyper_parameters.to_dict()
hyper_parameters["logger_config"] = {"type": JsonLogger, "logdir": checkpoint_path}
hyper_parameters["env_config"] = {
    "id": env_name + "NoFrameskip-v4",
}

In [7]:
# Set run object
run_name = "DDQN_%s" % env_name + "_PBER_%s" % run_name
env_example = env_creator(hyper_parameters["env_config"])
obs, _ = env_example.reset()
step = env_example.step(1)
print(env_example.action_space, env_example.observation_space)
print(env_example)
print("log path: %s; check_path: %s" % (log_path, checkpoint_path))
register_env("Atari", env_creator)

Discrete(4) Box(0, 255, (84, 84, 4), uint8)
<FrameStack<WarpFrame<FireResetEnv<EpisodicLifeEnv<MaxAndSkipEnv<NoopResetEnv<MonitorEnv<OrderEnforcing<PassiveEnvChecker<AtariEnv<BreakoutNoFrameskip-v4>>>>>>>>>>>
log path: /home/seventheli/data/BER/experiments/logging/Breakout/1712411533; check_path: /home/seventheli/data/BER/experiments/checkpoints/Breakout/1712411533


A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]


In [8]:
# Set Replay Buffer
replay_buffer_config = {
    **hyper_parameters["replay_buffer_config"],
    "type": MultiAgentPrioritizedBlockReplayBuffer,
    "obs_space": env_example.observation_space,
    "action_space": env_example.action_space,
    "sub_buffer_size": 8,
    "worker_side_prioritization": False,
    "replay_buffer_shards_colocated_with_driver": True,
    "rollout_fragment_length": hyper_parameters["rollout_fragment_length"],
}
hyper_parameters["replay_buffer_config"] = replay_buffer_config
hyper_parameters["optimizer"] = {"num_replay_buffer_shards": 1}

In [9]:
# Set Trainer
trainer = DDQNWithMPBERAndLogging(config=hyper_parameters, env="Atari")

2024-04-06 14:52:13,855	WARNING deprecation.py:50 -- DeprecationWarning: `rllib/algorithms/simple_q/` has been deprecated. Use `rllib_contrib/simple_q/` instead. This will raise an error in the future!
2024-04-06 14:52:13,857	WARNING deprecation.py:50 -- DeprecationWarning: `algo = Algorithm(env='Atari', ...)` has been deprecated. Use `algo = AlgorithmConfig().environment('Atari').build()` instead. This will raise an error in the future!
/home/seventheli/anaconda3/envs/ber_gym_28/lib/python3.9/site-packages/ray/rllib/utils/from_config.py:197: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  object_ = constructor(*ctor_args, **ctor_kwargs)
2024-04-06 14:52:14,018	WARNING deprecation.py:50 -- DeprecationWarning: `ray.

In [10]:
# Common setup
checkpoint_path = str(checkpoint_path)
# Save initial configuration
with open(os.path.join(checkpoint_path, "%s_config.pyl" % run_name), "wb") as f:
    _ = trainer.config.to_dict()
    pickle.dump(_, f)

In [11]:
mlflow_run = mlflow.start_run(run_name=run_name,
                              tags={"mlflow.user": "10900+3090"})
# Log parameters
mlflow.log_params(hyper_parameters["replay_buffer_config"])
to_log = ['double_q', 'dueling', 'lr', 'n_step', 'num_steps_sampled_before_learning_starts',
          'rollout_fragment_length', 'target_network_update_freq', 'train_batch_size', 'min_sample_timesteps_per_iteration']
mlflow.log_params(
    {key: hyper_parameters[key] for key in to_log})

In [12]:
keys_to_extract_sam = {"episode_reward_max", "episode_reward_min", "episode_reward_mean"}
keys_to_extract_sta = {"num_agent_steps_sampled", "num_agent_steps_trained"}
keys_to_extract_buf = {"add_batch_time_ms", "replay_time_ms", "update_priorities_time_ms"}

In [13]:
mlflow.pytorch.log_model(trainer.get_policy().model, run_name)
model_uri = "runs:/%s/model_name" % mlflow_run.info.run_id
mlflow.register_model(model_uri, run_name, tags={"episode" : 0})

/home/seventheli/anaconda3/envs/ber_gym_28/lib/python3.9/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
Successfully registered model 'DDQN_Breakout_PBER_1712411533'.
2024/04/06 14:52:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DDQN_Breakout_PBER_1712411533, version 1
Created version '1' of model 'DDQN_Breakout_PBER_1712411533'.


<ModelVersion: aliases=[], creation_timestamp=1712411543275, current_stage='None', description='', last_updated_timestamp=1712411543275, name='DDQN_Breakout_PBER_1712411533', run_id='3e2083f6abe44a6690c76913146e474f', run_link='', source='s3://jo-mlflow-ber/1/3e2083f6abe44a6690c76913146e474f/artifacts/model_name', status='READY', status_message='', tags={'episode': '0'}, user_id='', version='1'>

In [14]:
for i in tqdm(range(0, setting.log.max_run), ascii=True):
    result = trainer.train()
    time_used = result["time_total_s"]
    try:
        sampler = result.get("sampler_results", {}).copy()
        eva = result.get("evaluation", {}).copy()
        info = result.get("info", {}).copy()
        sam = {key: sampler[key] for key in keys_to_extract_sam if key in sampler}
        sta = {key: info[key] for key in keys_to_extract_sta if key in info}
        buf = flatten_dict(trainer.local_replay_buffer.stats())
        buf["est_size_gb"] = buf["est_size_bytes"] /1e9
        result['buffer'] = buf
        time_usage = info.get("learner", {}).get("time_usage", {})
        if eva:
            eva = {"eval_" + key: eva[key] for key in keys_to_extract_sam if key in eva}
        mlflow.log_metrics({**sam, **sta, **buf, **time_usage, **eva}, step=result["episodes_total"])
        if i % (setting.log.log * 50) == 0:
            trainer.save_checkpoint(checkpoint_path)
            mlflow.pytorch.log_model(trainer.get_policy().model, run_name)
            mlflow.register_model(model_uri, run_name, tags={
                "episode" : result["episodes_total"],
                "reward": sam["episode_reward_mean"],
            })
            mlflow.log_artifacts(log_path)
            mlflow.log_artifacts(checkpoint_path)
        if i % 10 == 0:
            tqdm.write("episode %d ; " % result["episodes_total"] +  " ".join(["%s : %f8" % (i, j)for i, j in sam.items()]))
    except FunctionTimedOut:
        tqdm.write("logging failed")
    except MlflowException:
        tqdm.write("logging failed")
    except ConnectionClosedError:
        tqdm.write("logging failed")
    with open(os.path.join(log_path, str(i) + ".json"), "w") as f:
        result["config"] = None
        json.dump(convert_np_arrays(result), f)
    if time_used >= setting.log.max_time:
        break

  0%|          | 0/20000 [00:00<?, ?it/s]

2024-04-06 14:52:23,669	WARNING multi_agent_prioritized_replay_buffer.py:215 -- Adding batches with column `weights` to this buffer while providing weights as a call argument to the add method results in the column being overwritten.
Registered model 'DDQN_Breakout_PBER_1712411533' already exists. Creating a new version of this model...
2024/04/06 14:53:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DDQN_Breakout_PBER_1712411533, version 2
Created version '2' of model 'DDQN_Breakout_PBER_1712411533'.


episode 63 ; episode_reward_min : 0.0000008 episode_reward_mean : 1.1428578 episode_reward_max : 9.0000008


2024-04-06 14:59:00,599	WARNING deprecation.py:50 -- DeprecationWarning: `ray.rllib.execution.train_ops.multi_gpu_train_one_step` has been deprecated. This will raise an error in the future!


episode 648 ; episode_reward_min : 0.0000008 episode_reward_mean : 1.1400008 episode_reward_max : 6.0000008
episode 1240 ; episode_reward_min : 0.0000008 episode_reward_mean : 1.4100008 episode_reward_max : 7.0000008
episode 1705 ; episode_reward_min : 0.0000008 episode_reward_mean : 4.7200008 episode_reward_max : 20.0000008
episode 2011 ; episode_reward_min : 0.0000008 episode_reward_mean : 9.0100008 episode_reward_max : 23.0000008
episode 2263 ; episode_reward_min : 1.0000008 episode_reward_mean : 11.3500008 episode_reward_max : 28.0000008
episode 2492 ; episode_reward_min : 4.0000008 episode_reward_mean : 15.9900008 episode_reward_max : 38.0000008
episode 2698 ; episode_reward_min : 6.0000008 episode_reward_mean : 17.2000008 episode_reward_max : 46.0000008
episode 2892 ; episode_reward_min : 8.0000008 episode_reward_mean : 18.8200008 episode_reward_max : 40.0000008
episode 3074 ; episode_reward_min : 6.0000008 episode_reward_mean : 20.0900008 episode_reward_max : 39.0000008
episode 

...

Registered model 'DDQN_Breakout_PBER_1712411533' already exists. Creating a new version of this model...
2024/04/08 21:03:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DDQN_Breakout_PBER_1712411533, version 4
Created version '4' of model 'DDQN_Breakout_PBER_1712411533'.


episode 12078 ; episode_reward_min : 2.0000008 episode_reward_mean : 126.9900008 episode_reward_max : 375.0000008
episode 12165 ; episode_reward_min : 16.0000008 episode_reward_mean : 184.5900008 episode_reward_max : 385.0000008
episode 12258 ; episode_reward_min : 1.0000008 episode_reward_mean : 147.7800008 episode_reward_max : 413.0000008
episode 12348 ; episode_reward_min : 8.0000008 episode_reward_mean : 118.0600008 episode_reward_max : 412.0000008
episode 12439 ; episode_reward_min : 8.0000008 episode_reward_mean : 152.1800008 episode_reward_max : 379.0000008
episode 12534 ; episode_reward_min : 10.0000008 episode_reward_mean : 139.3600008 episode_reward_max : 398.0000008
episode 12629 ; episode_reward_min : 11.0000008 episode_reward_mean : 150.3900008 episode_reward_max : 406.0000008
episode 12713 ; episode_reward_min : 19.0000008 episode_reward_mean : 159.6600008 episode_reward_max : 406.0000008
episode 12803 ; episode_reward_min : 18.0000008 episode_reward_mean : 130.2600008 ep

Registered model 'DDQN_Breakout_PBER_1712411533' already exists. Creating a new version of this model...
2024/04/10 00:20:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DDQN_Breakout_PBER_1712411533, version 5
Created version '5' of model 'DDQN_Breakout_PBER_1712411533'.


episode 16562 ; episode_reward_min : 1.0000008 episode_reward_mean : 141.9400008 episode_reward_max : 404.0000008
episode 16657 ; episode_reward_min : 4.0000008 episode_reward_mean : 145.8500008 episode_reward_max : 390.0000008
episode 16745 ; episode_reward_min : 6.0000008 episode_reward_mean : 160.2200008 episode_reward_max : 387.0000008
episode 16842 ; episode_reward_min : 10.0000008 episode_reward_mean : 132.6800008 episode_reward_max : 390.0000008
episode 16938 ; episode_reward_min : 2.0000008 episode_reward_mean : 113.8900008 episode_reward_max : 382.0000008
episode 17033 ; episode_reward_min : 2.0000008 episode_reward_mean : 144.3400008 episode_reward_max : 369.0000008
episode 17126 ; episode_reward_min : 7.0000008 episode_reward_mean : 125.1000008 episode_reward_max : 385.0000008
episode 17216 ; episode_reward_min : 10.0000008 episode_reward_mean : 162.9700008 episode_reward_max : 395.0000008
episode 17308 ; episode_reward_min : 1.0000008 episode_reward_mean : 131.9900008 episo


KeyboardInterrupt



In [ ]:
mlflow.log_artifacts(log_path)
mlflow.log_artifacts(checkpoint_path)

In [ ]:
mlflow.end_run()